In [11]:
# Importação das bibliotecas necessárias
import random
import math
import numpy
from deap import base
from deap import creator
from deap import tools
from deap import algorithms

In [12]:
# Configuração do Problema Bin Packing (Empacotamento)
NUM_ITENS = 50           # Quantidade de itens a serem empacotados
CAPACIDADE_BIN = 100     # Capacidade máxima de cada caixa (bin)

# Gerando pesos aleatórios para os itens (usando semente fixa para ser reprodutível)
random.seed(42)
PESOS = [random.randint(10, 50) for _ in range(NUM_ITENS)]

print("Pesos dos itens:", PESOS)

Pesos dos itens: [50, 17, 11, 27, 25, 24, 18, 16, 44, 15, 47, 37, 12, 11, 15, 23, 24, 42, 48, 11, 45, 22, 44, 36, 24, 38, 47, 27, 10, 20, 37, 31, 27, 19, 23, 31, 16, 15, 34, 16, 32, 32, 48, 26, 12, 39, 44, 17, 34, 15]


In [15]:
# Definição da representação por permutação + decodificação
# Em vez de evoluir diretamente a caixa de cada item, evoluímos a ordem dos itens.
# Depois, a heurística First Fit decide onde cada item entra.
for name in ["FitnessMin", "Individual"]:
    if hasattr(creator, name):
        delattr(creator, name)

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)


def first_fit_decode(order):
    """Recebe uma permutação dos índices dos itens e retorna as caixas preenchidas pela regra First Fit."""
    bins = []
    for item_idx in order:
        peso = PESOS[item_idx]
        colocado = False

        for b_idx in range(len(bins)):
            if sum(bins[b_idx]) + peso <= CAPACIDADE_BIN:
                bins[b_idx].append(peso)
                colocado = True
                break

        if not colocado:
            bins.append([peso])

    return bins


def evaluate_bin_packing(individual):
    """Fitness: número de caixas + penalidade por excesso."""
    bins = first_fit_decode(individual)
    num_bins_usados = len(bins)
    penalidade = 0

    for caixa in bins:
        peso_total = sum(caixa)
        if peso_total > CAPACIDADE_BIN:
            penalidade += (peso_total - CAPACIDADE_BIN) * 1000

    fitness = num_bins_usados + penalidade
    return fitness,


def first_fit_decreasing_order():
    """Ordem por peso decrescente, que funciona bem em Bin Packing."""
    return creator.Individual(sorted(range(NUM_ITENS), key=lambda i: PESOS[i], reverse=True))


def best_fit_decreasing_order():
    """Consegue uma ordem pela regra Best Fit Decreasing."""
    ordem = []
    bins = []

    for item_idx in sorted(range(NUM_ITENS), key=lambda i: PESOS[i], reverse=True):
        peso = PESOS[item_idx]
        melhor_bin = None
        melhor_sobra = None

        for b_idx, caixa in enumerate(bins):
            sobra = CAPACIDADE_BIN - (sum(caixa) + peso)
            if sobra >= 0 and (melhor_sobra is None or sobra < melhor_sobra):
                melhor_bin = b_idx
                melhor_sobra = sobra

        if melhor_bin is None:
            bins.append([peso])
        else:
            bins[melhor_bin].append(peso)

        ordem.append(item_idx)

    return creator.Individual(ordem)


def random_permutation():
    return creator.Individual(random.sample(range(NUM_ITENS), NUM_ITENS))


def weighted_order():
    """Ordem baseada em pesos, mas com leve embaralhamento para introduzir diversidade."""
    ordem = sorted(range(NUM_ITENS), key=lambda i: PESOS[i], reverse=True)
    if len(ordem) > 1:
        for _ in range(5):
            i, j = random.sample(range(len(ordem)), 2)
            ordem[i], ordem[j] = ordem[j], ordem[i]
    return creator.Individual(ordem)


def build_initial_population(size):
    """População inicial melhorada: FFD, BFD, aleatórias e ordenadas por peso."""
    pop = []
    qtd_por_tipo = size // 4

    for _ in range(qtd_por_tipo):
        pop.append(first_fit_decreasing_order())
        pop.append(best_fit_decreasing_order())
        pop.append(random_permutation())
        pop.append(weighted_order())

    while len(pop) < size:
        pop.append(random_permutation())

    return pop[:size]

In [16]:
toolbox = base.Toolbox()

toolbox.register("individual", random_permutation)
toolbox.register("population", list)
toolbox.register("evaluate", evaluate_bin_packing)
toolbox.register("mate", tools.cxOrdered)
toolbox.register("mutate", tools.mutShuffleIndexes, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

In [ ]:
# Função de avaliação e regra de decodificação
# A busca agora atua sobre a ordem dos itens, e a viabilidade é criada pela Decodificação First Fit.
# Isso reduz muito o espaço de busca inútil e torna o processo mais eficiente.

In [ ]:
# Operadores genéticos adaptados para permutação
# cxOrdered troca ordem preservando estrutura.
# mutShuffleIndexes troca posições do vetor, mantendo a permutação válida.

In [17]:
def main():
    random.seed(64)

    pop = build_initial_population(300)

    hof = tools.HallOfFame(1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", numpy.mean)
    stats.register("std", numpy.std)
    stats.register("min", numpy.min)
    stats.register("max", numpy.max)

    pop, log = algorithms.eaSimple(pop, toolbox, cxpb=0.8, mutpb=0.25, ngen=300,
                                   stats=stats, halloffame=hof, verbose=True)

    return pop, log, hof

In [18]:
if __name__ == "__main__":
    pop, log, hof = main()
    best_order = hof[0]
    best_bins = first_fit_decode(best_order)

    print("\n--- RESULTADO FINAL ---")
    print("Melhor ordem encontrada:", best_order)
    print("Fitness do Melhor Indivíduo:", best_order.fitness.values[0])
    print(f"Número de Caixas Utilizadas: {len(best_bins)}\n")

    for idx, caixa in enumerate(best_bins):
        print(f"Caixa {idx:02d}: Peso = {sum(caixa)} / {CAPACIDADE_BIN} | Itens = {caixa}")

gen	nevals	avg    	std    	min	max
0  	300   	14.9767	0.15096	14 	15 
1  	242   	14.9467	0.239072	14 	16 
2  	259   	14.9333	0.262467	14 	16 
3  	252   	14.9333	0.249444	14 	15 
4  	257   	14.9333	0.249444	14 	15 
5  	268   	14.95  	0.217945	14 	15 
6  	259   	14.96  	0.241661	14 	16 
7  	239   	14.9833	0.172401	14 	16 
8  	252   	14.9867	0.182087	14 	16 
9  	259   	15.01  	0.207926	14 	16 
10 	258   	15.0133	0.215613	14 	16 
11 	250   	15.02  	0.198997	14 	16 
12 	264   	15.0233	0.171626	14 	16 
13 	254   	15.01  	0.0994987	15 	16 
14 	257   	15.02  	0.14     	15 	16 
15 	253   	15.01  	0.0994987	15 	16 
16 	244   	15.01  	0.0994987	15 	16 
17 	256   	15     	0        	15 	15 
18 	248   	15.01  	0.0994987	15 	16 
19 	246   	15     	0        	15 	15 
20 	262   	14.9967	0.0576387	14 	15 
21 	257   	15.0033	0.0576387	15 	16 
22 	242   	15.0067	0.081377 	15 	16 
23 	243   	15     	0        	15 	15 
24 	265   	15     	0        	15 	15 
25 	250   	15     	0        	15 	15 
26 	258   	15    

In [19]:
# --- CÁLCULO DA EFICIÊNCIA ---
if __name__ == "__main__":
    soma_total_pesos = sum(PESOS)
    minimo_teorico_caixas = math.ceil(soma_total_pesos / CAPACIDADE_BIN)
    caixas_reais_usadas = len(best_bins)

    print("\n========================================")
    print("      ANÁLISE DE EFICIÊNCIA (ACURÁCIA)  ")
    print("========================================\n")
    print(f"Soma total de todos os pesos: {soma_total_pesos}")
    print(f"Mínimo teórico matemático de caixas: {minimo_teorico_caixas}")
    print(f"Caixas realmente utilizadas pelo algoritmo: {caixas_reais_usadas}\n")

    if caixas_reais_usadas == minimo_teorico_caixas:
        print("✅ Ótimo encontrado!")
    else:
        eficiencia = (minimo_teorico_caixas / caixas_reais_usadas) * 100
        print("✅ Solução válida")
        print(f"🎯 Eficiência da Otimização ('Acurácia'): {eficiencia:.2f}%")
        if eficiencia >= 90:
            print("\n🔥 Muito boa! Perto do ótimo.")
        else:
            print("\n⚠️ Ainda há espaço para melhorar com mais ajustes na mutação e na população inicial.")


      ANÁLISE DE EFICIÊNCIA (ACURÁCIA)  

Soma total de todos os pesos: 1378
Mínimo teórico matemático de caixas: 14
Caixas realmente utilizadas pelo algoritmo: 14

✅ Ótimo encontrado!
